#1. 对话历史优化案列（采取只保留N轮对话历史、加系统消息

In [2]:
import os

from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

# 定义函数，指保留N轮对话历史，默认2轮
def keep_recent_messages(messages, max_pairs=2):
    "1.保留系统消息"""
    sys_message = [m for m in messages if m["role"] == "system"]
    # 2.分离拿到其它消息
    other_messages = [m for m in messages if m["role"] != "system"]
    # 3.获取N轮对话历史，截取最后N*2条消息
    recent_messages = other_messages[-(max_pairs*2):]
    return sys_message + recent_messages



# 加载配置文件，存在相同key采用当前覆盖
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE   = os.getenv("DEEPSEEK_BASE_URL")
DEEPSEEK_MODEL_NAME   = os.getenv("DEEPSEEK_MODEL")

model = ChatDeepSeek(
    model_name=DEEPSEEK_MODEL_NAME,
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_API_BASE,
)

# 初始化
long_conversation = [
    {"role": "system", "content": "你是 Python 导师"}
]
# 第 1 轮
long_conversation.append({"role": "user", "content": "什么是列表？用一句解释"})
r1 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r1.content})

# 第 2 轮
long_conversation.append({"role": "user", "content": "列表和元组有什么区别？用一句解释"})
r2 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r2.content})

# 第 3 轮
long_conversation.append({"role": "user", "content": "什么是字典呢？用一句解释"})
r3 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r3.content})
print(f"原始消息数: {len(long_conversation)}")

# 优化：只保留最近 2 轮
optimized = keep_recent_messages(long_conversation, max_pairs=2)
print(f"优化后消息数: {len(optimized)}")
print(f"保留的内容: system + 最近2轮对话")
# 添加新的用户问题
optimized.append({"role": "user", "content": "我第一个问题问的是什么？"})
# 使用优化后的历史
response = model.invoke(optimized)
print(f"\nAI 回复: {response.content}")

原始消息数: 7
优化后消息数: 5
保留的内容: system + 最近2轮对话

AI 回复: 你问的第一个问题是：“列表和元组有什么区别？用一句解释”


2.content_blocks案列

In [19]:
import base64
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from dotenv import load_dotenv
import os
load_dotenv(override=True)

DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL   = os.getenv("DASHSCOPE_BASE_URL")


model = init_chat_model(
    model="openai:qwen-vl-plus",
    api_key = DASHSCOPE_API_KEY,
    base_url = DASHSCOPE_BASE_URL,
)


def encode_image(img_path):
    """将一张本地图片转换成 Base64 编码的 Data URI 字符串,方便在文本中嵌入图片数据"""
    with open(img_path, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode("utf-8")

# 图像路径
img_path = "test01.jpg"
# 获取图像base64编码字符串
base64_image = encode_image(img_path)
response = model.invoke([
    # HumanMessage(content=[
    #     {"type": "text", "text": "这张图里面是啥？"},
    #     {
    #         "type": "image_url",
    #         "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"},
    #     },
    # ])

    # 最新统一写法
    HumanMessage(
        content_blocks=[
        {'type': 'text', 'text': '这张图里有什么？'},
        {
        'type': 'image',
        'base64': base64_image,
        'mime_type': 'image/jpg',
        }
        ]
    )
])

print(response.content)

这张图片展示了一只狗的特写，背景模糊，焦点集中在狗的头部和上半身。以下是对图片中细节的详细描述：

### 1. **狗的品种**
   - 这只狗看起来像是一只边境牧羊犬（Border Collie）。它的毛色主要是白色，带有黑色的斑块，尤其是在头部和耳朵周围。
   - 边境牧羊犬以其聪明、活泼和忠诚著称，通常被用作牧羊犬或家庭宠物。

### 2. **狗的表情**
   - 狗的嘴巴微微张开，舌头露在外面，看起来像是在微笑或喘气。这种表情给人一种愉快、放松的感觉。
   - 它的眼睛明亮有神，似乎在注视着某个方向，显得非常专注和警觉。

### 3. **毛发**
   - 狗的毛发非常浓密且蓬松，尤其是颈部和背部的毛发显得特别柔软和有光泽。
   - 毛色以白色为主，耳朵和头部有一些黑色的斑块，形成了鲜明的对比。

### 4. **耳朵**
   - 狗的耳朵竖立着，形状较大，边缘略显卷曲。耳朵的颜色是黑白相间的，与整体毛色一致。
   - 耳朵的姿势表明它可能正在倾听周围的动静。

### 5. **背景**
   - 背景是一片模糊的草地或田野，颜色偏暖色调，可能是阳光照射下的自然环境。
   - 背景的虚化处理使得狗的形象更加突出，同时也营造出一种宁静、自然的氛围。

### 6. **光线**
   - 光线柔和，可能是傍晚或清晨的阳光，给画面增添了一种温暖的色调。
   - 光线从狗的侧面照射过来，突出了毛发的质感和细节。

### 7. **整体感觉**
   - 这张图片给人一种温馨、愉悦的感觉，狗的表情和姿态传达出一种快乐和满足的情绪。
   - 通过细腻的拍摄手法，摄影师成功地捕捉到了狗的神态和毛发的细节，使观者能够感受到它的活力和可爱。

总结来说，这张图片展示了一只边境牧羊犬在自然环境中的可爱瞬间，背景的虚化和柔和的光线进一步增强了画面的艺术感和情感表达。
